
# Phase 3 — Classical Time Series Models  
## Beginner-friendly, practical, and engaging

**Course:** Data Science with Python  
**Module:** Time Series Analysis  
**Phase 3 Focus:** Classical forecasting models  
**Suggested duration:** 6 hours  

---

## Why this phase matters

In Phase 1, students learned the foundations of time series.  
In Phase 2, students learned how to build forecasting features and use machine learning.

Now we turn to the classical forecasting world.

This phase focuses on models that were designed specifically for time series, especially:

- moving averages and smoothing ideas
- AR
- MA
- ARIMA
- SARIMA

The goal is not to turn students into statisticians.  
The goal is to make them **comfortable, practical, and conceptually clear**.

---

## Learning goals

By the end of this notebook, students should be able to:

- explain the difference between classical forecasting models and ML-based forecasting
- understand autoregression in intuitive terms
- understand the basic ideas behind AR, MA, ARIMA, and SARIMA
- explain the meaning of the ARIMA parameters `(p, d, q)`
- explain seasonality in SARIMA
- use train/test split for classical forecasting
- fit and evaluate simple ARIMA and SARIMA models in Python
- compare classical models with a baseline forecast
- interpret forecasts and residual behavior at a beginner level

---

## Suggested teaching structure for a 6-hour session

### Hour 1
- Why classical models exist
- Review of trend, seasonality, and stationarity
- Introduce the dataset

### Hour 2
- Autoregression intuition
- Moving average intuition
- AR, MA, and ARIMA ideas

### Hour 3
- Train/test split for forecasting
- Baseline forecast
- First ARIMA model

### Hour 4
- Meaning of p, d, q
- Compare a few ARIMA settings
- Forecast evaluation

### Hour 5
- Seasonal forecasting
- SARIMA intuition
- Fit a SARIMA model

### Hour 6
- Compare models
- Interpret results
- Practical lessons and exercises


In [ ]:

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline



# Part 1: Load a monthly time series dataset

We will use a monthly airline-style passenger dataset built for teaching.

This kind of data is good for classical forecasting because it has:
- long-term trend
- clear seasonality
- regular monthly frequency

**File used:** `time_series_phase3_monthly_airline_style.csv`

Target column:
- `passengers`


In [ ]:

# Load dataset
df = pd.read_csv("/mnt/data/time_series_phase3_monthly_airline_style.csv")

# Show first rows
df.head()


In [ ]:

# Convert date column and set as index
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

df.head()


In [ ]:

# Basic shape and info
print("Rows and columns:", df.shape)
print()
print(df.dtypes)


In [ ]:

# Plot the full series
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["passengers"])
plt.xlabel("Date")
plt.ylabel("Passengers")
plt.title("Monthly Passengers Over Time")
plt.show()



## First observations

Ask students:
- Is there a trend?
- Does the series seem seasonal?
- Are the highs and lows repeating?
- Does the overall level grow over time?

This kind of visual reading is a core time series skill.



# Part 2: Why classical models?

Machine learning forecasting often works by:
- creating lag features
- creating calendar features
- using a regression model

Classical time series models do something different.

They try to model the structure of the series itself.

That means they directly use ideas like:
- dependence on past values
- dependence on past errors
- differencing to remove trend
- explicit seasonal structure

These models are often very useful when:
- the series is regular
- the history is not too short
- seasonality is clear
- we want interpretable forecasting structure



# Part 3: Quick review of stationarity

Classical models often work better when the series is closer to **stationary**.

In beginner terms, stationarity means the series behaves more consistently over time.

A strongly trending series is often not stationary.

A strongly seasonal series may also need transformation before modeling.


In [ ]:

# First difference
df["diff_1"] = df["passengers"].diff()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df.index, df["passengers"])
axes[0].set_title("Original Series")
axes[0].set_ylabel("Passengers")

axes[1].plot(df.index, df["diff_1"])
axes[1].set_title("First Difference")
axes[1].set_ylabel("Difference")

plt.xlabel("Date")
plt.show()



## Why differencing matters

First differencing means:

\[
y_t - y_{t-1}
\]

This often helps remove trend.

In ARIMA, the `d` parameter tells us how many times we difference the series.



# Part 4: Decomposition review

Before modeling, it is often helpful to visualize trend and seasonality more explicitly.


In [ ]:

# Additive decomposition
decomp = seasonal_decompose(df["passengers"], model="additive", period=12)
decomp.plot()
plt.show()



## Teaching point

This decomposition reinforces why classical models are useful:
- the series has trend
- the series has seasonality
- there is still some irregular movement left over



# Part 5: Autoregression intuition

A very important classical idea is:

> the past helps predict the future

Autoregression means we predict the current value using previous values of the same series.

A simple AR(1) idea looks like:

\[
y_t = c + \phi_1 y_{t-1} + \varepsilon_t
\]

This means:
- today's value depends partly on yesterday's value
- plus some random noise


In [ ]:

# Visual demo: current month vs previous month
df["lag_1"] = df["passengers"].shift(1)

demo = df.dropna()

plt.figure(figsize=(6, 5))
plt.scatter(demo["lag_1"], demo["passengers"], alpha=0.7)
plt.xlabel("Previous Month Passengers")
plt.ylabel("Current Month Passengers")
plt.title("Autoregression Intuition: Current vs Previous Month")
plt.show()



## What students should notice

If points show a clear relationship, then the past contains information about the present.

That is the core intuition of autoregression.



# Part 6: Moving average intuition

In classical time series models, **MA** does not mean moving average in the everyday plotting sense.

Instead, MA means:

> the current value depends on past forecast errors

A simple MA(1) idea looks like:

\[
y_t = \mu + \varepsilon_t + \theta_1 \varepsilon_{t-1}
\]

This is harder for beginners, so keep the intuition simple:

- AR uses past values
- MA uses past errors



## Beginner summary so far

- **AR** = use past values
- **MA** = use past errors
- **ARIMA** = combine AR, differencing, and MA



# Part 7: ARIMA intuition

ARIMA stands for:

- **AR** = autoregressive part
- **I** = integrated part (differencing)
- **MA** = moving average part

The notation is:

\[
ARIMA(p, d, q)
\]

Where:

- `p` = number of AR terms
- `d` = number of differences
- `q` = number of MA terms



## What do the parameters mean in plain English?

### `p`
How many past values we use

### `d`
How many times we difference the series to reduce trend

### `q`
How many past error terms we use

---

## Example

`ARIMA(1,1,1)` means:
- use 1 lag of the series
- difference once
- use 1 lag of the forecast error



# Part 8: Chronological train/test split

We will forecast the final 12 months.

This is a common and realistic setup for monthly forecasting.


In [ ]:

# Split into train and test
train = df.loc[: "2024-12-01", "passengers"]
test = df.loc["2025-01-01":, "passengers"]

print("Train length:", len(train))
print("Test length :", len(test))
print("Train end   :", train.index.max())
print("Test start  :", test.index.min())



# Part 9: Baseline forecast

Before fitting classical models, we should compare against a simple baseline.

A useful seasonal baseline for monthly data is:

> predict this month using the value from 12 months ago


In [ ]:

# Seasonal naive baseline: use value from 12 months ago
baseline_forecast = df["passengers"].shift(12).loc[test.index]

baseline_forecast


In [ ]:

# Forecast metric helper
def forecast_report(y_true, y_pred, label="Model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"--- {label} ---")
    print(f"MAE : {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"MAPE: {mape:.2f}%")
    print()


In [ ]:

# Evaluate the baseline
forecast_report(test, baseline_forecast, label="Seasonal Naive Baseline")


In [ ]:

# Plot train, test, and baseline
plt.figure(figsize=(12, 4))
plt.plot(train.index, train.values, label="Train")
plt.plot(test.index, test.values, label="Test")
plt.plot(test.index, baseline_forecast.values, label="Seasonal Naive Forecast", linestyle="--")
plt.xlabel("Date")
plt.ylabel("Passengers")
plt.title("Baseline Forecast")
plt.legend()
plt.show()



## Important lesson

Always compare your classical model to a simple baseline.

A more advanced model should justify its complexity.



# Part 10: Fit a first ARIMA model

We will begin with a simple ARIMA model:

`ARIMA(1,1,1)`

This is a common starting point for beginners.


In [ ]:

# Fit ARIMA(1,1,1)
arima_111 = ARIMA(train, order=(1, 1, 1))
arima_111_fit = arima_111.fit()

print(arima_111_fit.summary())



## How to read the summary at a beginner level

Do not try to explain every number.

Focus on:
- the model used
- estimated coefficients
- AIC / BIC as rough fit indicators
- whether the model converged successfully

For beginners, it is enough to understand that the model has been fitted and the coefficients have been estimated from the training data.


In [ ]:

# Forecast the test period
arima_111_forecast = arima_111_fit.forecast(steps=len(test))

forecast_report(test, arima_111_forecast, label="ARIMA(1,1,1)")


In [ ]:

# Plot ARIMA forecast
plt.figure(figsize=(12, 4))
plt.plot(train.index, train.values, label="Train")
plt.plot(test.index, test.values, label="Test")
plt.plot(test.index, arima_111_forecast.values, label="ARIMA(1,1,1) Forecast", linestyle="--")
plt.xlabel("Date")
plt.ylabel("Passengers")
plt.title("ARIMA(1,1,1) Forecast")
plt.legend()
plt.show()



# Part 11: Compare a few ARIMA settings

In practice, we often try more than one ARIMA specification.

We will compare a few simple candidates:

- ARIMA(1,1,0)
- ARIMA(0,1,1)
- ARIMA(1,1,1)
- ARIMA(2,1,2)


In [ ]:

# Compare a few ARIMA models
orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,2)]
rows = []

for order in orders:
    model = ARIMA(train, order=order)
    fit = model.fit()
    pred = fit.forecast(steps=len(test))

    rows.append({
        "order": str(order),
        "AIC": fit.aic,
        "MAE": mean_absolute_error(test, pred),
        "RMSE": mean_squared_error(test, pred) ** 0.5,
        "MAPE": np.mean(np.abs((test - pred) / test)) * 100
    })

pd.DataFrame(rows).sort_values("RMSE")



## Teaching point

Two useful lessons here:

1. Model choice matters  
2. Lower AIC and lower forecast error are related, but not always perfectly aligned

For beginners, the main message is:
> compare a few reasonable models instead of trusting one guess blindly



# Part 12: Why ARIMA may still struggle

Our series clearly has seasonality.

Plain ARIMA handles trend and short-term structure, but it does not model seasonality as directly as SARIMA does.

That is why seasonal models matter.



# Part 13: SARIMA intuition

SARIMA extends ARIMA by adding seasonal terms.

The notation is often written as:

\[
SARIMA(p,d,q)\times(P,D,Q,s)
\]

Where:
- `(p,d,q)` = regular ARIMA part
- `(P,D,Q,s)` = seasonal part
- `s` = seasonal length

For monthly data with yearly seasonality:
- `s = 12`



## Plain English meaning

If we use:

`SARIMA(1,1,1) x (1,1,1,12)`

that means:
- regular AR, differencing, and MA terms
- plus seasonal AR, differencing, and MA terms
- with a 12-month seasonal cycle



# Part 14: Fit a first SARIMA model

We will start with:

`SARIMA(1,1,1) x (1,1,1,12)`


In [ ]:

# Fit SARIMA
sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_fit = sarima_model.fit(disp=False)

print(sarima_fit.summary())


In [ ]:

# Forecast using SARIMA
sarima_forecast = sarima_fit.forecast(steps=len(test))

forecast_report(test, sarima_forecast, label="SARIMA(1,1,1)x(1,1,1,12)")


In [ ]:

# Plot SARIMA forecast
plt.figure(figsize=(12, 4))
plt.plot(train.index, train.values, label="Train")
plt.plot(test.index, test.values, label="Test")
plt.plot(test.index, sarima_forecast.values, label="SARIMA Forecast", linestyle="--")
plt.xlabel("Date")
plt.ylabel("Passengers")
plt.title("SARIMA Forecast")
plt.legend()
plt.show()



## What students should notice

Because the series is seasonal, SARIMA often does a better job matching the recurring pattern than plain ARIMA.



# Part 15: Compare baseline, ARIMA, and SARIMA


In [ ]:

comparison = pd.DataFrame([
    {
        "model": "Seasonal Naive Baseline",
        "MAE": mean_absolute_error(test, baseline_forecast),
        "RMSE": mean_squared_error(test, baseline_forecast) ** 0.5,
        "MAPE": np.mean(np.abs((test - baseline_forecast) / test)) * 100
    },
    {
        "model": "ARIMA(1,1,1)",
        "MAE": mean_absolute_error(test, arima_111_forecast),
        "RMSE": mean_squared_error(test, arima_111_forecast) ** 0.5,
        "MAPE": np.mean(np.abs((test - arima_111_forecast) / test)) * 100
    },
    {
        "model": "SARIMA(1,1,1)x(1,1,1,12)",
        "MAE": mean_absolute_error(test, sarima_forecast),
        "RMSE": mean_squared_error(test, sarima_forecast) ** 0.5,
        "MAPE": np.mean(np.abs((test - sarima_forecast) / test)) * 100
    }
])

comparison.sort_values("RMSE")


In [ ]:

# Plot all forecasts together
plt.figure(figsize=(12, 5))
plt.plot(train.index, train.values, label="Train")
plt.plot(test.index, test.values, label="Test")
plt.plot(test.index, baseline_forecast.values, label="Seasonal Naive", linestyle="--")
plt.plot(test.index, arima_111_forecast.values, label="ARIMA(1,1,1)", linestyle="--")
plt.plot(test.index, sarima_forecast.values, label="SARIMA", linestyle="--")
plt.xlabel("Date")
plt.ylabel("Passengers")
plt.title("Forecast Comparison")
plt.legend()
plt.show()



## A very practical lesson

The best model depends on the structure of the data.

- if seasonality is strong, seasonal models can help a lot
- a simple baseline may already be hard to beat
- not every series needs a very complex model



# Part 16: Residual intuition

Residuals are:

\[
\text{actual} - \text{predicted}
\]

Good forecasting residuals should ideally:
- be centered around zero
- avoid obvious pattern
- avoid systematic trend


In [ ]:

# Residual plot for SARIMA forecast
sarima_residuals = test - sarima_forecast

plt.figure(figsize=(10, 4))
plt.plot(sarima_residuals.index, sarima_residuals.values)
plt.axhline(0, linestyle="--")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.title("SARIMA Forecast Residuals")
plt.show()



## Beginner interpretation

If residuals still show a strong pattern, the model may be missing something.

For this course, students do not need advanced residual diagnostics yet.  
They just need to understand the idea.



# Part 17: Practical lessons about classical models

### 1. Classical models are built for time series structure
They directly model lag relationships and differencing.

### 2. Stationarity matters
Differencing is often an important step.

### 3. Seasonality matters
If the pattern repeats, seasonal models may help a lot.

### 4. Baselines still matter
Do not skip baseline comparison.

### 5. Model choice is empirical
Try a few reasonable options and compare results.



# Part 18: Hands-on exercises

## Exercise 1
Fit `ARIMA(1,1,0)` and compare with `ARIMA(0,1,1)`.

## Exercise 2
Try `ARIMA(2,1,1)`.

## Exercise 3
Fit a different SARIMA model, such as:
- `SARIMA(0,1,1)x(0,1,1,12)`

## Exercise 4
Explain in plain English:
- AR
- MA
- differencing
- ARIMA
- SARIMA

## Exercise 5
Why is the seasonal naive forecast a strong baseline for this dataset?

## Exercise 6
Which model would you choose here, and why?


In [ ]:

# Starter code for Exercise 3
sarima_alt = SARIMAX(
    train,
    order=(0, 1, 1),
    seasonal_order=(0, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_alt_fit = sarima_alt.fit(disp=False)
sarima_alt_forecast = sarima_alt_fit.forecast(steps=len(test))

forecast_report(test, sarima_alt_forecast, label="Alternative SARIMA")



# Part 19: Key takeaways

- Classical models are designed specifically for time series forecasting
- AR uses past values
- MA uses past errors
- ARIMA combines AR, differencing, and MA
- SARIMA extends ARIMA to handle seasonality
- The parameters `(p, d, q)` describe the non-seasonal structure
- The seasonal part adds seasonal dynamics
- Chronological train/test split is essential
- Baseline comparison is essential
- Practical forecasting means trying sensible models and comparing them honestly



## End of Phase 3

A natural Phase 4 could cover:
- multi-step forecasting
- Prophet or other business-friendly tools
- advanced evaluation
- model comparison across ML and classical approaches
- practical deployment issues
